In [6]:
import glob
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

In [7]:
SEQ_LEN = 20

BATCH_SIZE = 64

EPOCHS = 20

LEARNING_RATE = 0.001

HIDDEN_SIZE = 64

NUM_LAYERS = 1

SEARCH_RADIUS_KM = 50.0

RANDOM_SEED = 42

FEATURE_COLS = [
    "lat",
    "lon",
    "baroaltitude"
]

In [8]:
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

In [9]:
csv_files = glob.glob("../data/states_2017-06-05-0*.csv")

print(f"Found {len(csv_files)} CSV files")

df = pd.concat(
    (pd.read_csv(csv_file) for csv_file in csv_files),
    ignore_index=True
)

print(f"Total rows: {len(df)}")

print(df.columns.tolist())

Found 10 CSV files
Total rows: 13955628
['time', 'icao24', 'lat', 'lon', 'velocity', 'heading', 'vertrate', 'callsign', 'onground', 'alert', 'spi', 'squawk', 'baroaltitude', 'geoaltitude', 'lastposupdate', 'lastcontact']


In [10]:
required_cols = [
    "icao24",
    "callsign",
    "time",
    "lat",
    "lon",
    "baroaltitude"
]

missing = [
    col for col in required_cols
    if col not in df.columns
]

if missing:
    raise ValueError(
        f"Missing columns: {missing}"
    )

df["icao24"] = df["icao24"].astype(str).str.strip()

df["callsign"] = (
    df["callsign"]
    .astype(str)
    .str.strip()
    .str.upper()
)

for col in ["lat", "lon", "baroaltitude"]:
    df[col] = pd.to_numeric(
        df[col],
        errors="coerce"
    )

    df["time"] = pd.to_numeric(
    df["time"],
    errors="coerce"
)
    
df = df[
    (df["callsign"] != "") &
    (df["callsign"] != "NAN")
]

df = df.dropna(
    subset=[
        "icao24",
        "callsign",
        "time",
        "lat",
        "lon",
        "baroaltitude"
    ]
)

df = df.sort_values(
    ["icao24", "callsign", "time"]
).reset_index(drop=True)

print(f"Rows after cleaning: {len(df)}")
print(f"Unique aircraft: {df['icao24'].nunique()}")
print(f"Unique trajectories: {df.groupby(['icao24', 'callsign']).ngroups}")

Rows after cleaning: 8684253
Unique aircraft: 13492
Unique trajectories: 48072


In [11]:
trajectories = {
    key: group.to_dict("records")
    for key, group in df.groupby(
        ["icao24", "callsign"]
    )
}

first_key = next(iter(trajectories))

print("Trajectory:", first_key)
print("Number of points:", len(trajectories[first_key]))

print(trajectories[first_key][:2])

Trajectory: ('004022', 'GAA4571')
Number of points: 51
[{'time': 1496641240, 'icao24': '004022', 'lat': 24.9042205811, 'lon': 55.1472473145, 'velocity': 97.9391131796, 'heading': 301.686367688, 'vertrate': 15.93088, 'callsign': 'GAA4571', 'onground': False, 'alert': False, 'spi': False, 'squawk': nan, 'baroaltitude': 320.04, 'geoaltitude': 289.56, 'lastposupdate': 1496641238.19, 'lastcontact': 1496641239.29}, {'time': 1496641250, 'icao24': '004022', 'lat': 24.9050748146, 'lon': 55.1455112673, 'velocity': 97.9391131796, 'heading': 301.686367688, 'vertrate': 15.93088, 'callsign': 'GAA4571', 'onground': False, 'alert': False, 'spi': False, 'squawk': 7275.0, 'baroaltitude': 342.9, 'geoaltitude': 289.56, 'lastposupdate': 1496641240.23, 'lastcontact': 1496641246.36}]


In [12]:
trajectories = {
    key: trajectory
    for key, trajectory in trajectories.items()
    if len(trajectory) > SEQ_LEN
}

print(
    f"Trajectories with more than "
    f"{SEQ_LEN} points: {len(trajectories)}"
)

Trajectories with more than 20 points: 22521


In [13]:
random.seed(42)

items = list(trajectories.items())

random.shuffle(items)

n = len(items)

train_end = int(n * 0.70)

val_end = int(n * 0.90)

train = dict(items[:train_end])

val = dict(items[train_end:val_end])

test = dict(items[val_end:])

print(f"Train: {len(train)} trajectories")
print(f"Val:   {len(val)} trajectories")
print(f"Test:  {len(test)} trajectories")

Train: 15764 trajectories
Val:   4504 trajectories
Test:  2253 trajectories


In [14]:
def trajectory_to_array(trajectory):

    points = []

    for point in trajectory:

        points.append([
            point["lat"],
            point["lon"],
            point["baroaltitude"]
        ])

    return np.asarray(
        points,
        dtype=np.float32
    )

key = next(iter(train))

arr = trajectory_to_array(train[key])

print(arr.shape)
print(arr[:3])

(214, 3)
[[  -30.089218   117.044785 10363.2     ]
 [  -30.099197   117.02777  10363.2     ]
 [  -30.108591   117.01181  10363.2     ]]


In [15]:
train_points = []

for trajectory in train.values():

    arr = trajectory_to_array(trajectory)

    train_points.append(arr)

train_points = np.concatenate(
    train_points,
    axis=0
)

print("Training points:", len(train_points))

scaler = StandardScaler()

scaler.fit(train_points)

print("Mean:")
print(scaler.mean_)

print("\nScale:")
print(scaler.scale_)

Training points: 6061797
Mean:
[ 3.62674755e+01 -5.26934198e+00  8.08860132e+03]

Scale:
[  21.70744874   75.66913766 4009.28393674]


In [16]:
def create_sequence_dataset(
    trajectories,
    scaler,
    seq_len=20
):

    X = []
    y = []

    for trajectory in trajectories.values():

        arr = trajectory_to_array(trajectory)

        normalized = scaler.transform(arr)

        for i in range(
            len(normalized) - seq_len
        ):

            X.append(
                normalized[
                    i:i + seq_len
                ]
            )

            y.append(
                normalized[
                    i + seq_len
                ]
            )

    X = np.asarray(
        X,
        dtype=np.float32
    )

    y = np.asarray(
        y,
        dtype=np.float32
    )

    return X, y

X_train, y_train = create_sequence_dataset(
    train,
    scaler,
    SEQ_LEN
)

X_val, y_val = create_sequence_dataset(
    val,
    scaler,
    SEQ_LEN
)

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

print("X_val:", X_val.shape)
print("y_val:", y_val.shape)

X_train: (5746517, 20, 3)
y_train: (5746517, 3)
X_val: (1646649, 20, 3)
y_val: (1646649, 3)


In [17]:
train_dataset = TensorDataset(
    torch.tensor(X_train),
    torch.tensor(y_train)
)

val_dataset = TensorDataset(
    torch.tensor(X_val),
    torch.tensor(y_val)
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

In [18]:
class SimpleBiLSTM(nn.Module):

    def __init__(self):

        super().__init__()

        self.lstm = nn.LSTM(
            input_size=3,
            hidden_size=HIDDEN_SIZE,
            num_layers=NUM_LAYERS,
            batch_first=True,
            bidirectional=True
        )

        self.fc = nn.Linear(
            HIDDEN_SIZE * 2,
            3
        )

    def forward(self, x):

        output, _ = self.lstm(x)

        output = output[:, -1, :]

        return self.fc(output)
    
device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print("Using:", device)

model = SimpleBiLSTM().to(device)

criterion = nn.MSELoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=LEARNING_RATE
)

Using: cuda


In [19]:
train_losses = []
val_losses = []

for epoch in range(EPOCHS):

    # -------------------------
    # Training
    # -------------------------

    model.train()

    train_loss = 0.0

    for X_batch, y_batch in train_loader:

        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()

        prediction = model(X_batch)

        loss = criterion(
            prediction,
            y_batch
        )

        loss.backward()

        optimizer.step()

        train_loss += (
            loss.item() *
            X_batch.size(0)
        )

    train_loss /= len(train_loader.dataset)

    # -------------------------
    # Validation
    # -------------------------

    model.eval()

    val_loss = 0.0

    with torch.no_grad():

        for X_batch, y_batch in val_loader:

            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            prediction = model(X_batch)

            loss = criterion(
                prediction,
                y_batch
            )

            val_loss += (
                loss.item() *
                X_batch.size(0)
            )

    val_loss /= len(val_loader.dataset)

    train_losses.append(train_loss)
    val_losses.append(val_loss)

    print(
        f"Epoch {epoch + 1:02d}/{EPOCHS} "
        f"| Train: {train_loss:.6f} "
        f"| Val: {val_loss:.6f}"
    )

Epoch 01/20 | Train: 0.006884 | Val: 0.006341
Epoch 02/20 | Train: 0.006310 | Val: 0.006328
Epoch 03/20 | Train: 0.006246 | Val: 0.006214
Epoch 04/20 | Train: 0.006209 | Val: 0.006260
Epoch 05/20 | Train: 0.006196 | Val: 0.006251
Epoch 06/20 | Train: 0.006175 | Val: 0.006228
Epoch 07/20 | Train: 0.006174 | Val: 0.006253
Epoch 08/20 | Train: 0.006164 | Val: 0.006215


KeyboardInterrupt: 

In [ ]:
plt.figure(figsize=(10, 5))

plt.plot(
    train_losses,
    label="Training"
)

plt.plot(
    val_losses,
    label="Validation"
)

plt.xlabel("Epoch")
plt.ylabel("MSE Loss")
plt.title("BiLSTM Training")

plt.legend()

plt.grid(True)

plt.show()

In [ ]:
def haversine_distance(
    lat1,
    lon1,
    lat2,
    lon2
):

    R = 6371.0

    lat1 = np.radians(lat1)
    lat2 = np.radians(lat2)

    delta_lat = np.radians(
        lat2 - lat1
    )

    delta_lon = np.radians(
        lon2 - lon1
    )

    a = (
        np.sin(delta_lat / 2) ** 2
        +
        np.cos(lat1)
        * np.cos(lat2)
        * np.sin(delta_lon / 2) ** 2
    )

    c = 2 * np.arctan2(
        np.sqrt(a),
        np.sqrt(1 - a)
    )

    return R * c

In [ ]:
def search_area_bounds(
    lat,
    lon,
    radius_km
):

    # Latitude:
    # approximately 111.32 km per degree

    delta_lat = radius_km / 111.32

    # Longitude:
    # 111.32 * cos(latitude)

    longitude_km_per_degree = (
        111.32 * np.cos(
            np.radians(lat)
        )
    )

    delta_lon = (
        radius_km /
        longitude_km_per_degree
    )

    return (
        lat - delta_lat,
        lat + delta_lat,
        lon - delta_lon,
        lon + delta_lon
    )

lat = -23.55
lon = -46.63

bounds = search_area_bounds(
    lat,
    lon,
    50
)

print(bounds)

In [ ]:
def constrain_to_search_area(
    center_lat,
    center_lon,
    prediction_lat,
    prediction_lon,
    radius_km
):

    distance = haversine_distance(
        center_lat,
        center_lon,
        prediction_lat,
        prediction_lon
    )

    # Already inside the search area
    if distance <= radius_km:
        return (
            prediction_lat,
            prediction_lon
        )

    # Convert coordinates to radians

    lat1 = np.radians(center_lat)
    lon1 = np.radians(center_lon)

    lat2 = np.radians(prediction_lat)
    lon2 = np.radians(prediction_lon)

    # Bearing from center to prediction

    y = (
        np.sin(lon2 - lon1)
        * np.cos(lat2)
    )

    x = (
        np.cos(lat1)
        * np.sin(lat2)
        -
        np.sin(lat1)
        * np.cos(lat2)
        * np.cos(lon2 - lon1)
    )

    bearing = np.arctan2(y, x)

    # Radius of Earth

    R = 6371.0

    angular_distance = (
        radius_km / R
    )

    # Destination point

    new_lat = np.arcsin(
        np.sin(lat1)
        * np.cos(angular_distance)
        +
        np.cos(lat1)
        * np.sin(angular_distance)
        * np.cos(bearing)
    )

    new_lon = (
        lon1
        +
        np.arctan2(
            np.sin(bearing)
            * np.sin(angular_distance)
            * np.cos(lat1),
            np.cos(angular_distance)
            -
            np.sin(lat1)
            * np.sin(new_lat)
        )
    )

    return (
        np.degrees(new_lat),
        np.degrees(new_lon)
    )

In [ ]:
def predict_next_points(
    model,
    trajectory,
    scaler,
    seq_len=20,
    search_radius_km=50
):

    model.eval()

    trajectory = trajectory_to_array(
        trajectory
    )

    if len(trajectory) <= seq_len:
        return None

    normalized = scaler.transform(
        trajectory
    )

    predictions = []
    constrained_predictions = []
    actuals = []
    centers = []
    errors = []
    raw_errors = []

    with torch.no_grad():

        for i in range(
            seq_len,
            len(trajectory)
        ):

            # -------------------------
            # Last 20 REAL observations
            # -------------------------

            sequence = normalized[
                i - seq_len:i
            ]

            sequence = torch.tensor(
                sequence,
                dtype=torch.float32
            ).unsqueeze(0)

            sequence = sequence.to(device)

            # -------------------------
            # Model prediction
            # -------------------------

            prediction = model(
                sequence
            )

            prediction = (
                prediction
                .squeeze(0)
                .cpu()
                .numpy()
            )

            # Back to lat/lon/altitude

            prediction = scaler.inverse_transform(
                prediction.reshape(1, -1)
            )[0]

            # -------------------------
            # Last known REAL position
            # -------------------------

            center_lat = trajectory[
                i - 1, 0
            ]

            center_lon = trajectory[
                i - 1, 1
            ]

            # -------------------------
            # Raw prediction error
            # -------------------------

            raw_error = haversine_distance(
                center_lat,
                center_lon,
                prediction[0],
                prediction[1]
            )

            # -------------------------
            # Constrain prediction
            # -------------------------

            constrained_lat, constrained_lon = (
                constrain_to_search_area(
                    center_lat,
                    center_lon,
                    prediction[0],
                    prediction[1],
                    search_radius_km
                )
            )

            constrained_prediction = prediction.copy()

            constrained_prediction[0] = (
                constrained_lat
            )

            constrained_prediction[1] = (
                constrained_lon
            )

            # -------------------------
            # Actual next position
            # -------------------------

            actual = trajectory[i]

            # -------------------------
            # Final error
            # -------------------------

            error = haversine_distance(
                actual[0],
                actual[1],
                constrained_prediction[0],
                constrained_prediction[1]
            )

            # -------------------------
            # Store
            # -------------------------

            predictions.append(
                prediction
            )

            constrained_predictions.append(
                constrained_prediction
            )

            actuals.append(
                actual
            )

            centers.append([
                center_lat,
                center_lon
            ])

            raw_errors.append(
                raw_error
            )

            errors.append(
                error
            )

    return {
        "predicted": np.asarray(
            predictions
        ),

        "constrained_predicted": np.asarray(
            constrained_predictions
        ),

        "actual": np.asarray(
            actuals
        ),

        "centers": np.asarray(
            centers
        ),

        "raw_errors": np.asarray(
            raw_errors
        ),

        "errors": np.asarray(
            errors
        ),

        "search_radius_km":
            search_radius_km
    }

In [ ]:
test_key = next(iter(test))

test_trajectory = test[test_key]

print("Trajectory:", test_key)

print(
    "Number of points:",
    len(test_trajectory)
)

result = predict_next_points(
    model,
    test_trajectory,
    scaler,
    seq_len=SEQ_LEN,
    search_radius_km=SEARCH_RADIUS_KM
)

print(
    "Mean error:",
    np.mean(result["errors"]),
    "km"
)

print(
    "Median error:",
    np.median(result["errors"]),
    "km"
)

print(
    "Maximum error:",
    np.max(result["errors"]),
    "km"
)

outside = (
    result["raw_errors"]
    > SEARCH_RADIUS_KM
)

print(
    "Predictions outside search area:",
    np.sum(outside)
)

print(
    "Percentage outside:",
    100 * np.mean(outside),
    "%"
)

In [ ]:
all_errors = []
all_raw_errors = []

trajectory_results = {}

for key, trajectory in test.items():

    result = predict_next_points(
        model,
        trajectory,
        scaler,
        seq_len=SEQ_LEN,
        search_radius_km=SEARCH_RADIUS_KM
    )

    if result is None:
        continue

    trajectory_results[key] = result

    all_errors.extend(
        result["errors"]
    )

    all_raw_errors.extend(
        result["raw_errors"]
    )

all_errors = np.asarray(
    all_errors
)

all_raw_errors = np.asarray(
    all_raw_errors
)

In [ ]:
print("=" * 40)

print("SEARCH AREA EVALUATION")

print("=" * 40)

print(
    f"Search radius: "
    f"{SEARCH_RADIUS_KM:.1f} km"
)

print(
    f"Predictions: "
    f"{len(all_errors)}"
)

print()

print(
    f"Mean error: "
    f"{np.mean(all_errors):.2f} km"
)

print(
    f"Median error: "
    f"{np.median(all_errors):.2f} km"
)

print(
    f"Maximum error: "
    f"{np.max(all_errors):.2f} km"
)

print(
    f"90th percentile: "
    f"{np.percentile(all_errors, 90):.2f} km"
)

print(
    f"95th percentile: "
    f"{np.percentile(all_errors, 95):.2f} km"
)

In [ ]:
print("=" * 40)
print("RAW MODEL")
print("=" * 40)

print(
    f"Mean:   "
    f"{np.mean(all_raw_errors):.2f} km"
)

print(
    f"Median: "
    f"{np.median(all_raw_errors):.2f} km"
)

print(
    f"Max:    "
    f"{np.max(all_raw_errors):.2f} km"
)

print("=" * 40)
print("SEARCH-AREA CONSTRAINED")
print("=" * 40)

print(
    f"Mean:   "
    f"{np.mean(all_errors):.2f} km"
)

print(
    f"Median: "
    f"{np.median(all_errors):.2f} km"
)

print(
    f"Max:    "
    f"{np.max(all_errors):.2f} km"
)

In [ ]:
key = next(iter(trajectory_results))

result = trajectory_results[key]

actual = result["actual"]
predicted = result["constrained_predicted"]
centers = result["centers"]

plt.figure(figsize=(10, 7))

plt.plot(
    actual[:, 1],
    actual[:, 0],
    label="Actual trajectory"
)

plt.scatter(
    predicted[:, 1],
    predicted[:, 0],
    s=10,
    label="BiLSTM prediction"
)

plt.scatter(
    centers[:, 1],
    centers[:, 0],
    s=5,
    alpha=0.4,
    label="Last known position"
)

plt.xlabel("Longitude")
plt.ylabel("Latitude")

plt.title(
    f"BiLSTM Trajectory Prediction\n"
    f"{key[0]} / {key[1]}"
)

plt.legend()

plt.grid(True)

plt.show()